# RAG Pipeline — LangChain ingestion + OpenAI + PostgreSQL/pgvector

Same tutorial flow as before, but with **your** stack:

| Step | Tool |
|------|------|
| Load PDFs / split text | **LangChain** (`PyMuPDFLoader`, `RecursiveCharacterTextSplitter`) |
| Embeddings | **OpenAI** via `langchain_openai.OpenAIEmbeddings` |
| Vector store | **PostgreSQL + pgvector** (SQL you already know) |
| LLM answer | **OpenAI** via `langchain_openai.ChatOpenAI` |

Requires: `OPENAI_API_KEY` in a `.env`, Postgres with `pgvector`, and a DB (default `vectortutorialdb`).

In [28]:
import os
import json
import uuid
from pathlib import Path
from typing import List, Dict, Any

import psycopg2
from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
print("OPENAI_API_KEY set:", os.getenv("OPENAI_API_KEY") is not None)

OPENAI_API_KEY set: True


## 1. Load PDFs (LangChain)

Each PDF page becomes a LangChain `Document` with `page_content` + `metadata`.
We add `source_file` / `file_type` so retrieval results stay traceable.

In [ ]:
def process_all_pdfs(pdf_directory: str):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    for pdf_file in pdf_files:
        print(f"processing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
        except Exception as e:
            print(f"x Error loading {pdf_file.name}: {e}")

    print(f"\nLoaded {len(all_documents)} page-documents from {len(pdf_files)} PDFs")
    return all_documents


all_pdf_documents = process_all_pdfs("../data")
all_pdf_documents

## 2. Chunk documents (LangChain)

Embedding whole pages is coarse. `RecursiveCharacterTextSplitter` breaks text into overlapping chunks so retrieval can return the most relevant *section*.

- `chunk_size=500` — ~characters per chunk
- `chunk_overlap=200` — overlap keeps sentences from being cut off awkwardly

In [ ]:
def split_documents(documents, chunk_size=500, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


chunks = split_documents(all_pdf_documents)
chunks[:2]

## 3. Embeddings (OpenAI)

Replaces the tutorial's local `SentenceTransformer` (`all-MiniLM-L6-v2`, 384 dims).

`OpenAIEmbeddings` defaults to a 1536-dim model. Same idea as your `Notebooks/pgvector.ipynb`: text → list of floats.

In [ ]:
class EmbeddingManager:
    """Thin wrapper around LangChain's OpenAI embeddings."""

    def __init__(self, model: str = "text-embedding-3-small"):
        # text-embedding-3-small → 1536 dimensions by default
        self.model_name = model
        self.embeddings = OpenAIEmbeddings(model=model)
        print(f"Using OpenAI embedding model: {model}")

    def generate_embeddings(self, texts: List[str]) -> List[List[float]]:
        print(f"generating embeddings for {len(texts)} texts")
        vectors = self.embeddings.embed_documents(texts)
        print(f"generated {len(vectors)} embeddings, dim={len(vectors[0])}")
        return vectors

    def embed_query(self, query: str) -> List[float]:
        return self.embeddings.embed_query(query)

    def get_embedding_dimension(self) -> int:
        # probe once so CREATE TABLE uses the right vector(N)
        return len(self.embed_query("dimension probe"))


embedding_manager = EmbeddingManager()
print("embedding dim:", embedding_manager.get_embedding_dimension())

## 4. Vector store (PostgreSQL + pgvector)

Replaces ChromaDB. Same mental model as your earlier notebook:

1. `CREATE EXTENSION vector`
2. Table with `content`, `embedding vector(N)`, plus metadata
3. `INSERT` after embedding
4. Search with `ORDER BY embedding <-> query LIMIT k`

Update `DATABASE_URL` if your DB name/user/password differ.

In [ ]:
# Same style as Notebooks/pgvector.ipynb — change if needed
DATABASE_URL = os.getenv(
    "DATABASE_URL",
    "dbname=vectortutorialdb user=postgres password=postgres host=localhost port=5432",
)


class VectorStore:
    """Postgres + pgvector store for LangChain Document chunks."""

    def __init__(
        self,
        connection_string: str = DATABASE_URL,
        table_name: str = "langchain_chunks",
        embedding_dim: int = 1536,
    ):
        self.connection_string = connection_string
        self.table_name = table_name
        self.embedding_dim = embedding_dim
        self._initialize_store()

    def _connect(self):
        return psycopg2.connect(self.connection_string)

    def _initialize_store(self):
        conn = self._connect()
        cur = conn.cursor()
        try:
            cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
            cur.execute(
                f"""
                CREATE TABLE IF NOT EXISTS {self.table_name} (
                    id TEXT PRIMARY KEY,
                    content TEXT NOT NULL,
                    metadata JSONB,
                    embedding vector({self.embedding_dim})
                )
                """
            )
            conn.commit()
            print(f"Ready: table `{self.table_name}` (vector({self.embedding_dim}))")
        finally:
            cur.close()
            conn.close()

    def clear(self):
        """Wipe rows so re-running the notebook does not duplicate chunks."""
        conn = self._connect()
        cur = conn.cursor()
        cur.execute(f"DELETE FROM {self.table_name}")
        conn.commit()
        cur.close()
        conn.close()
        print(f"Cleared `{self.table_name}`")

    def add_documents(self, documents: List[Any], embeddings: List[List[float]]):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        conn = self._connect()
        cur = conn.cursor()

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            cur.execute(
                f"""
                INSERT INTO {self.table_name} (id, content, metadata, embedding)
                VALUES (%s, %s, %s::jsonb, %s::vector)
                """,
                (doc_id, doc.page_content, json.dumps(metadata), embedding),
            )

        conn.commit()
        cur.close()
        conn.close()
        print(f"Inserted {len(documents)} chunks into `{self.table_name}`")

    def similarity_search(
        self, query_embedding: List[float], top_k: int = 5
    ) -> List[Dict[str, Any]]:
        """Nearest neighbors by L2 distance (`<->`), same as your pgvector notebook."""
        conn = self._connect()
        cur = conn.cursor()
        cur.execute(
            f"""
            SELECT id, content, metadata, embedding <-> %s::vector AS distance
            FROM {self.table_name}
            ORDER BY embedding <-> %s::vector
            LIMIT %s
            """,
            (query_embedding, query_embedding, top_k),
        )
        rows = cur.fetchall()
        cur.close()
        conn.close()

        results = []
        for rank, (doc_id, content, metadata, distance) in enumerate(rows, start=1):
            results.append(
                {
                    "id": doc_id,
                    "content": content,
                    "metadata": metadata,
                    "distance": float(distance),
                    # Rough similarity for display (not cosine; L2-based)
                    "similarity_score": 1 / (1 + float(distance)),
                    "rank": rank,
                }
            )
        return results


vectorstore = VectorStore(
    embedding_dim=embedding_manager.get_embedding_dimension()
)
vectorstore

## 5. Embed chunks and insert

`clear()` first so re-running cells does not stack duplicate rows.

In [ ]:
vectorstore.clear()

texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks, embeddings)

## 6. Retriever

1. Embed the user question with the **same** OpenAI model
2. Ask Postgres for the closest chunks with `<->`
3. Return text + metadata for the LLM

In [ ]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self, query: str, top_k: int = 5, score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        query_embedding = self.embedding_manager.embed_query(query)
        results = self.vector_store.similarity_search(query_embedding, top_k=top_k)

        filtered = [
            doc for doc in results if doc["similarity_score"] >= score_threshold
        ]
        if not filtered:
            print("No documents found above threshold")
        return filtered


rag_retriever = RAGRetriever(vectorstore, embedding_manager)
rag_retriever.retrieve("what is python programming", top_k=3)

## 7. Answer with OpenAI (RAG)

Replaces the tutorial's Groq LLM.

Pattern:
1. Retrieve top chunks
2. Stuff them into a prompt as context
3. Ask ChatOpenAI to answer **only** from that context

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer using only the provided context. "
            "If the context is not enough, say you do not know.",
        ),
        (
            "human",
            "Context:\n{context}\n\nQuestion: {question}",
        ),
    ]
)


def answer_with_rag(question: str, top_k: int = 3) -> str:
    docs = rag_retriever.retrieve(question, top_k=top_k)
    context = "\n\n---\n\n".join(
        f"[source={d['metadata'].get('source_file', '?')} | rank={d['rank']}]\n{d['content']}"
        for d in docs
    )

    print("Retrieved sources:")
    for d in docs:
        print(
            f"  #{d['rank']} {d['metadata'].get('source_file')} "
            f"(distance={d['distance']:.4f})"
        )

    chain = rag_prompt | llm
    response = chain.invoke({"context": context, "question": question})
    return response.content


question = "what is python programming"
print("\nAnswer:\n", answer_with_rag(question))

## Mental model

```
PDFs
  → LangChain loaders          (Document objects)
  → LangChain text splitter    (chunks)
  → OpenAI embeddings          (vectors)
  → PostgreSQL / pgvector      (persistent store)
  → embed question + <->       (retrieve top-k)
  → ChatOpenAI + context       (final answer)
```

Compared to the original tutorial:
- **Chroma** → your Postgres table + `<->`
- **SentenceTransformers** → OpenAI embeddings
- **Groq** → OpenAI chat
- **LangChain loaders/splitters** stay — that is the ingestion layer you want to keep learning